# 03 · Role discovery — SOM (Drachen et al. replication)
Pipeline per side: standardize → **Self-Organizing Map** → k-means on the SOM
codebook → assign each player-round the role of its best-matching unit →
**manually inspect and name** the emergent roles → show **stability** across
retrainings. This mirrors Drachen, Canossa & Yannakakis (CIG 2009) step for
step — the methodological lineage of the thesis.

**This is the locked version for the current 76-demo dataset.** k and the
role names below were decided by inspecting the k-tables and profiles
(decision log inline). The fit is deterministic (fixed seed + fixed config),
so *Run all* reproduces the identical taxonomy. If you **add demos** or
**change config**, cluster NUMBERING can shuffle — re-verify `FINAL_NAMES`
against the signature cheat-sheet in §3 before trusting it.

In [ ]:
# --- bootstrap (identical in every notebook) ---------------------------------
from google.colab import drive
drive.mount('/content/drive')
%pip -q install demoparser2 minisom

import sys
sys.path.insert(0, '/content/drive/MyDrive/cs2-btp/code')

from cs2btp import (config as cfg, manifest as mf, parsing, qc,
                    features as ft, roles, consistency as cons,
                    outcome as out, viz)
cfg.ensure_dirs()
print('pipeline ready | drive root =', cfg.DRIVE_ROOT)

## 1 · Load features & standardize (per side)

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
feats = ft.load_all()
X_side = {}
for side in ('T','CT'):
    sub = (feats.loc[feats.side==side, cfg.CLUSTER_FEATURES]
           .replace([np.inf,-np.inf], np.nan).fillna(0.0))
    X_side[side] = StandardScaler().fit_transform(sub)
    print(side, X_side[side].shape)

## 2 · Choose k (number of roles) per side
Read the tables with two eyes at once: the silhouette elbow, AND
`min_cluster_share` not collapsing toward ~0 (a near-empty cluster = an
outlier pocket, not a role). Among statistically indistinguishable
candidates, prefer the smaller k — unless a larger k splits a dominant
cluster into two *orthogonal, interpretable* profiles (check before deciding).

In [ ]:
for side in ('T','CT'):
    print(f'--- {side} side ---')
    print(roles.select_k(X_side[side]).round(3).to_string(index=False))

In [ ]:
# DECISION (locked for this dataset):
#  T = 4 : silhouette peak (0.106 vs ~0.09 flat above), balanced shares.
#  CT = 5: k=4 has marginally better silhouette (0.108 vs 0.090) but k=5
#          splits the 48% rifler mega-cluster into two ORTHOGONAL archetypes
#          (Site Anchor vs Rotating Rifler, ~25% each) at balanced shares —
#          better construct validity + consistency resolution where the data
#          mass lies. k=4 is retained as a robustness variant (notebook 06).
K_BY_SIDE = {'T': 4, 'CT': 5}

## 3 · Fit final SOMs and inspect the emergent roles
Signature cheat-sheet for matching profiles to archetypes (match by
SIGNATURE, never by cluster number — numbering shuffles between fits):

* **AWPer** — `awp_share` spike (z ≈ +2.5…+3.6), high `buy_value_rel_team`, few shots.
* **Lurker (T)** — `dist_nearest_teammate` & `dist_team_centroid` ≈ +1.8, lowest `trade_presence`.
* **Aggressor** — high `opening_duel_involved`, earliest contact (negative z), lowest `time_alive_share`; on T it also absorbs eco rushes (crosstab in Appendix).
* **Support / Rotating Rifler** — mobile, carries utility (+0.4…+0.6 across nades), rifles, survives.
* **Site Anchor (CT)** — isolated (+0.6…+0.8 on both distance features), high `site_time_share`, low `zone_entropy`, low travel.
* **Eco/Save (CT)** — utility row uniformly strongly negative, low buy value (confirm with the Appendix crosstab).

In [ ]:
labeled, arte = roles.discover_roles(feats, K_BY_SIDE)
for side in ('T','CT'):
    print(f'--- {side}: heuristic suggestions (names are decided in §4) ---')
    print(arte[side]['names'])
    print(arte[side]['profile'].round(2))

In [ ]:
for side in ('T','CT'):
    viz.umatrix(arte[side]['som'], side)
    viz.role_radar(arte[side]['profile'], arte[side]['names'], side)

## 4 · Manual role naming (the human-in-the-loop step)
Names below were assigned by reading the §3 profiles against the
cheat-sheet. After applying, check the printed shares against the expected
values in the comment — a mismatch means the numbering shuffled.

In [ ]:
FINAL_NAMES = {
  'T':  {0: 'Aggressor', 1: 'AWPer', 2: 'Support Rifler', 3: 'Lurker'},
  'CT': {0: 'Site Anchor', 1: 'AWPer', 2: 'Rotating Rifler', 3: 'Eco/Save', 4: 'Aggressor'},
}
# expected shares for this dataset —
#   T : Aggressor ~43%, Support ~40%, Lurker ~11%, AWPer ~6%
#   CT: Rotator ~26%, Anchor ~25%, Eco ~21%, Aggressor ~18%, AWPer ~11%
labeled = roles.apply_manual_names(labeled, FINAL_NAMES)
labeled[['side','role_name']].value_counts()

In [ ]:
# regenerate the radar figures with the FINAL names (overwrites the
# suggestion-named versions saved in section 3) so figures/ matches the thesis
for side in ('T','CT'):
    viz.role_radar(arte[side]['profile'], FINAL_NAMES[side], side)

## 5 · Stability (the source paper's headline claim, replicated)
Global ARI across 8 bootstrap+seed retrainings, plus agreement with a plain
k-means baseline (~10–20 min at 100k SOM iterations). Then the per-cluster
view: global ARI is dominated by boundary redraws between large soft
clusters, so the per-cluster best-match Jaccard is the informative table —
expect specialists (AWPer ~0.9) ≫ positional/generalist roles (~0.5–0.7).

In [ ]:
for side in ('T','CT'):
    stab = roles.stability_check(X_side[side], K_BY_SIDE[side])
    lbl = labeled.loc[feats.side==side, 'role_id'].to_numpy()
    ari_km = roles.kmeans_baseline(X_side[side], K_BY_SIDE[side], lbl)
    print(f'{side}: stability ARI = {stab.ari.mean():.3f} '
          f'(min {stab.ari.min():.3f}) | vs k-means ARI = {ari_km:.3f}')
    stab.to_csv(cfg.ANALYSIS / f'stability_{side}.csv', index=False)

In [ ]:
import json

def per_cluster_stability(X, k, headline, n_runs=6, seed=123):
    rng = np.random.default_rng(seed)
    out = {c: [] for c in np.unique(headline)}
    for _ in range(n_runs):
        s = int(rng.integers(1e6))
        som = roles.fit_som(X, s)
        lab, _, _ = roles.som_labels(som, X, k, s)
        for c in out:
            m1 = headline == c
            best = max(((m1 & (lab == c2)).sum() / (m1 | (lab == c2)).sum())
                       for c2 in np.unique(lab))
            out[c].append(best)
    return {c: round(float(np.mean(v)), 2) for c, v in out.items()}

for side in ('T','CT'):
    lbl = labeled.loc[labeled.side==side, 'role_id'].to_numpy()
    res = per_cluster_stability(X_side[side], K_BY_SIDE[side], lbl)
    named = {FINAL_NAMES[side].get(int(c), str(c)): v for c, v in res.items()}
    print(side, named)
    with open(cfg.ANALYSIS / f'per_cluster_stability_{side}.json', 'w') as fh:
        json.dump(named, fh, indent=2)

## Appendix · Taxonomy diagnostics (thesis exhibits)
Everything below feeds the writeup: economy–role coupling, and external
(face) validity against roles the community assigns by eye.

In [ ]:
import pandas as pd
for side in ('T','CT'):
    d = labeled[labeled.side==side]
    print(f'--- {side}: role x buy_type ---')
    tab = pd.crosstab(d.role_name, d.buy_type, normalize='index').round(2)
    print(tab.to_string())
    tab.to_csv(cfg.ANALYSIS / f'role_buytype_crosstab_{side}.csv')

In [ ]:
# external validity: do star players land in their known roles?
for star in ['ZywOo', 'donk', 'm0NESY', 'broky']:
    sub = labeled[labeled.player_name.str.contains(star, case=False, na=False)]
    print(star, sub.groupby('side').role_name
          .agg(lambda s: s.value_counts(normalize=True).head(2).round(2).to_dict()).to_dict())

### Archive checklist (before moving to notebook 04)
Both k-tables · both profile tables · U-matrices · radar figures ·
`analysis/stability_*.csv` · `analysis/per_cluster_stability_*.json` ·
`analysis/role_buytype_crosstab_*.csv` · the face-validity printout ·
the k / naming decision comments in §2 and §4. Then run notebook 04 (Run all).